In [ ]:
п !pip install torchdeq
!unzip resnet_.zip
!unzip mdeq_.zip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchdeq import get_deq
import numpy as np
import os
import glob


In [ ]:
class DEQCell(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim, dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dim)
        )
    def forward(self, z, x):
        return torch.relu(self.conv(z) + x)

class MDEQSmall(nn.Module):
    def __init__(self, dim=64, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True)
        )
        self.cell = DEQCell(dim)
        self.deq = get_deq()
        self.deq_kwargs = {'solver': 'broyden', 'f_max_iter': 40, 'f_tol': 1e-3}
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(dim, num_classes))

        self.last_info = {}

    def forward(self, x):
        x_in = self.stem(x)
        z0 = torch.zeros_like(x_in)
        z_out, info = self.deq(lambda z: self.cell(z, x_in), z0, **self.deq_kwargs)

        # стан сольвера
        self.last_info['nstep'] = info['nstep'].float().mean().item()
        self.last_info['res'] = info.get('abs_lowest', info.get('lowest_res', torch.tensor(0.0))).mean().item()
        return self.head(z_out[0] if isinstance(z_out, list) else z_out)

class BalancedResNet(nn.Module):
    def __init__(self, dim=64, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True))

        self.layers = nn.Sequential(*[nn.Sequential(

            nn.Conv2d(dim, dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True),

            nn.Conv2d(dim, dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True)
        ) for _ in range(4)])

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(dim, num_classes))

    def forward(self, x):
        return self.head(self.layers(self.stem(x)))


def fgsm_attack(model, images, labels, eps):

    images = images.clone().detach().requires_grad_(True)

    outputs = model(images)
    loss = F.cross_entropy(outputs, labels)

    model.zero_grad()
    loss.backward()


    if images.grad is None:
        return images.detach()


    adv_images = images + eps * images.grad.sign()
    return torch.clamp(adv_images, -1, 1).detach()

def pgd_attack(model, images, labels, eps, alpha=0.01, steps=10):
    adv_images = images.clone().detach()

    for _ in range(steps):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = F.cross_entropy(outputs, labels)

        model.zero_grad()
        loss.backward()

        with torch.no_grad():
            if adv_images.grad is not None:
                adv_images = adv_images + alpha * adv_images.grad.sign()
                # Проекція на L-infinity куб
                delta = torch.clamp(adv_images - images, -eps, eps)
                adv_images = torch.clamp(images + delta, -1, 1).detach()
            else:
                break

    return adv_images



def get_best(model_class, folder, loader, device):
    paths = glob.glob(os.path.join(folder, "*.pth"))
    best_acc, best_path = 0, ""
    model = model_class().to(device)
    for p in paths:
        model.load_state_dict(torch.load(p, map_location=device))
        model.eval()
        correct = 0
        with torch.no_grad():
            for d, t in loader: correct += (model(d.to(device)).argmax(1) == t.to(device)).sum().item()
        acc = correct / len(loader.dataset)
        if acc > best_acc: best_acc, best_path = acc, p
    return best_path

# пошкодження даних, як ImageNet-C

def corrupt_data(images, corruption_type='gaussian_noise', severity=1):
    """Симуляція пошкоджень набору даних CIFAR-C."""

    if corruption_type == 'gaussian_noise':
        # Severity керує рівнем стандартного відхилення
        noise_level = [0.04, 0.06, 0.08, 0.12, 0.18][severity-1]
        noise = torch.randn_like(images) * noise_level
        return torch.clamp(images + noise, -1, 1)

    elif corruption_type == 'defocus_blur':
        # Використання Gaussian Blur, як для ImageNet-C defocus blur
        kernel_size = 2 * severity + 1
        blurrer = transforms.GaussianBlur(kernel_size=(kernel_size, kernel_size), sigma=(0.5 * severity))
        return blurrer(images)

    return images



def test_robustness_extended(model, loader, device, name,
                             attack=None, eps=0.0,
                             corruption=None, severity=1):
    model.eval()
    correct = 0
    total = 0
    steps_list = []
    residuals_list = []

    print(f"\n>>> Тестування {name} | Сценарій: {attack or corruption or 'Clean'} (eps={eps}, sev={severity})")

    for d, t in loader:
        d, t = d.to(device), t.to(device)

        if corruption:
            d = corrupt_data(d, corruption, severity)

        # (PGD/FGSM)
        if attack == 'fgsm':
            d = fgsm_attack(model, d, t, eps)
        elif attack == 'pgd':
            d = pgd_attack(model, d, t, eps)


        with torch.no_grad():
            outputs = model(d)
            correct += (outputs.argmax(1) == t).sum().item()
            total += t.size(0)

            # збір метрик збіжності для MDEQ
            if hasattr(model, 'last_info'):
                steps_list.append(model.last_info['nstep'])
                residuals_list.append(model.last_info['res'])

    acc = correct / total
    avg_steps = np.mean(steps_list) if steps_list else float('nan')
    avg_res = np.mean(residuals_list) if residuals_list else float('nan')

    print(f"Результат: Accuracy: {acc:.4f} | Avg Iters: {avg_steps:.2f} | Avg Residual: {avg_res:.6f}")
    return acc, avg_steps



In [4]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    loader = DataLoader(datasets.CIFAR10('./data', train=False, download=True, transform=tf), batch_size=100)


    best_r = get_best(BalancedResNet, "ResNet_Project_Checkpoints", loader, device)
    best_m = get_best(MDEQSmall, "DEQ_Project_Checkpoints", loader, device)
    print(f"best_resnet: {best_r}")
    print(f"best_mdeq: {best_m}")

    r_model = BalancedResNet().to(device)
    r_model.load_state_dict(torch.load(best_r, map_location=device))

    m_model = MDEQSmall().to(device)
    m_model.load_state_dict(torch.load(best_m, map_location=device))


    for sc in [{'attack':None, 'eps':0.0, 'noise':0.0}, {'attack':None, 'eps':0.0, 'noise':0.1},
               {'attack':'fgsm', 'eps':0.03, 'noise':0.0}, {'attack':'pgd', 'eps':0.03, 'noise':0.0}]:
        test_robustness(r_model, loader, device, "ResNet", **sc)
        test_robustness(m_model, loader, device, "MDEQ  ", **sc)


      device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    test_set = datasets.CIFAR10('./data', train=False, download=True, transform=tf)
    loader = DataLoader(test_set, batch_size=100, shuffle=False)

    # СЦЕНАРІЙ 1: Крива погіршення якості (Noise Robustness)
    print("\n=== ЕТАП 1: КРИВА ПОГІРШЕННЯ (GAUSSIAN NOISE SEVERITY) ===")
    for sev in range(1, 6):
        test_robustness_extended(r_model, loader, device, "ResNet", corruption='gaussian_noise', severity=sev)
        test_robustness_extended(m_model, loader, device, "MDEQ", corruption='gaussian_noise', severity=sev)

    # СЦЕНАРІЙ 2: Оцінка на пошкодженнях ImageNet-C (Defocus Blur)
    print("\n=== ЕТАП 2: ОЦІНКА НА DEFOCUS BLUR (IMAGENET-C STYLE) ===")
    test_robustness_extended(r_model, loader, device, "ResNet", corruption='defocus_blur', severity=3)
    test_robustness_extended(m_model, loader, device, "MDEQ", corruption='defocus_blur', severity=3)

    # СЦЕНАРІЙ 3: Суперечлива стійкість та збіжність сольвера
    print("\n=== ЕТАП 3: ADVERSARIAL ATTACKS & SOLVER CONVERGENCE ===")
    for eps in [0.01, 0.03, 0.07]:
        test_robustness_extended(r_model, loader, device, "ResNet", attack='pgd', eps=eps)
        test_robustness_extended(m_model, loader, device, "MDEQ", attack='pgd', eps=eps)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 24)


=== ЕТАП 1: КРИВА ПОГІРШЕННЯ (GAUSSIAN NOISE SEVERITY) ===

>>> Тестування ResNet | Сценарій: gaussian_noise (eps=0.0, sev=1)
Результат: Accuracy: 0.8153 | Avg Iters: nan | Avg Residual: nan

>>> Тестування MDEQ | Сценарій: gaussian_noise (eps=0.0, sev=1)
Результат: Accuracy: 0.6418 | Avg Iters: 37.86 | Avg Residual: 9.628359

>>> Тестування ResNet | Сценарій: gaussian_noise (eps=0.0, sev=2)
Результат: Accuracy: 0.7607 | Avg Iters: nan | Avg Residual: nan

>>> Тестування MDEQ | Сценарій: gaussian_noise (eps=0.0, sev=2)
Результат: Accuracy: 0.6290 | Avg Iters: 37.71 | Avg Residual: 9.897852

>>> Тестування ResNet | Сценарій: gaussian_noise (eps=0.0, sev=3)
Результат: Accuracy: 0.6743 | Avg Iters: nan | Avg Residual: nan

>>> Тестування MDEQ | Сценарій: gaussian_noise (eps=0.0, sev=3)
Результат: Accuracy: 0.5951 | Avg Iters: 37.71 | Avg Residual: 10.187659

>>> Тестування ResNet | Сценарій: gaussian_noise (eps=0.0, sev=4)
Результат: Accuracy: 0.4982 | Avg Iters: nan | Avg Residual: nan
